# ASHE earnings (workplace-based) → LAD-level table, 1997–2025

Pulls **ASHE workplace-analysis earnings at local-authority level** from the Nomis API and saves a tidy table: `clean/ashe_workplace_lad.csv`.

- **Source:** Nomis **NM_99_1** (ASHE, workplace analysis), **full-time workers** (`sex=8`). Per LA per year: **number of jobs**, **median**, **percentiles (10–90)**, for **gross weekly** (`pay=1`) and **gross hourly** (`pay=5`) pay, each with its **CV** (Nomis calls the measure *"Confidence"*; it is the coefficient of variation, %). GB only — **NI is not in NM_99_1**.
- **Workplace-based**: earnings of jobs *located in* each LA (residence-based, `NM_30_1`, is the companion notebook `ashe_residence.ipynb`; same structure and output columns).
- **Geography query:** `TYPE424` rather than the Nomis query-builder geography list, which was checked and **missed 29 of the 32 Scottish councils** (321 of 350 GB LAs).
- **Kept at LA level on purpose.** Medians/percentiles can't be summed or averaged across areas, so nothing is aggregated here. Each row is an LA unit *as published*, tagged with its **current LA** (`la_code_current`, via `lib/la_code_crosswalk.csv`) and **ITL2** (via `clean/geography_spine.csv`). Jobs + percentiles are the ingredients for the later jobs-weighted percentile interpolation to ITL2.

**Why several geography vintages.** Nomis files each year's estimates under the LA codes valid in a given boundary set, and no single set holds the whole history:

| Nomis geography | holds |
|---|---|
| `TYPE424` LADs (April 2023) | 2008–2025, but reorganised LAs only from when they existed in that set |
| `TYPE432` / `TYPE434` LADs (April 2021 / 2019) | e.g. East/West Suffolk 2018–21, North/West Northamptonshire 2020–21 |
| `TYPE434` / `TYPE464` LADs (April 2019 / pre-April 2015) | 1997–2007 (most LAs come from the April 2019 set) |
| **predecessor codes** (every old code in the crosswalk) | pre-merger districts, e.g. Cornwall's 6 districts pre-2008, Allerdale/Carlisle… pre-2022, Bournemouth/Poole pre-2018 |

**Stitching rule (§2b):**
1. A given LA code + year can appear in several vintages. They almost always agree; where they don't, the **typed vintage pulls are trusted over the code-queried predecessor pull** and the newest vintage with data wins. (Known case for workplace: the April 2019 set (`TYPE434`) holds **stale 2022 figures for almost every LA** — the other three sets agree with each other — so its 2022 values are never used; the newest set wins. Checked: every 2022 row in the output is from the April 2023 set.)
2. For each **current LA × year**: use the current LA itself if it has data; otherwise use its **predecessors** (via the crosswalk). Never both → **no area counted twice** (checked). Every unit nests in one ITL2 because reorganisations merged whole districts.
3. Predecessor units that **duplicate each other** — identical values, or the same area under successive codes (recodes) — are kept **once**, otherwise jobs would be double-weighted later.

**Known data limits (checked in §3):**
- **Suppressed cells are kept as NaN** with their Nomis `status` (`B` = CV > 20%, unreliable; `F` = sample 0–2; `Q` = missing). So CV never exceeds 20 — use the ONS bands in `cv_band`: ≤5 precise, 5–10 reasonably precise, 10–20 use with caution.
- **P90 is suppressed for ~90% of LAs, P80 for ~15%** — the upper tail is effectively unavailable at LA level.
- **Number of jobs is rounded to the nearest 1,000.**
- **ASHE series breaks:** 2004 (methodology change; 1997–2003 are on the earlier basis), 2006 (questionnaire change), 2011 (SOC 2010), 2021 (SOC 2020). Treat level shifts across these with care.
- **Northern Ireland:** the pre-2015 set lists NI's 26 old districts (`95AA`…`95ZZ`), but every value is empty, so they are dropped (reported as "no ITL2" in §2) and NI stays missing.
- **Isles of Scilly** is missing in every year (tiny sample); Orkney and a few small LAs in occasional years.
- **⚠ Must page the API.** The Nomis API caps each request at 25,000 rows and truncates silently, so every fetch pages and is checked for completeness.

In [1]:
import time
from datetime import date
from pathlib import Path
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
CROSSWALK = BASE / "lib" / "la_code_crosswalk.csv"
SPINE     = BASE / "clean" / "geography_spine.csv"
RAW_DIR   = BASE / "raw" / "ashe"
OUT       = BASE / "clean" / "ashe_workplace_lad.csv"

# Nomis LA geography vintages, newest first (= priority when the same code-year is in several)
VINTAGES = {"TYPE424": "LAD 2023", "TYPE432": "LAD 2021", "TYPE434": "LAD 2019", "TYPE464": "LAD pre-2015"}

QUERY = ("https://www.nomisweb.co.uk/api/v01/dataset/NM_99_1.data.csv"
         "?geography={geo}"
         "&sex=8"                      # full-time workers
         "&item=1,2,6...15"            # jobs, median, P10-P90
         "&pay=1,5"                    # gross weekly, gross hourly
         "&measures=20100,20701"       # value, confidence (= CV %)
         "&select=date_name,geography_code,geography_name,pay_name,item_name,"
         "measures_name,obs_value,obs_status")

xw    = pd.read_csv(CROSSWALK, dtype=str)
remap = dict(zip(xw.old_code, xw.new_code))
spine = pd.read_csv(SPINE)
lad2itl = dict(zip(spine.la_code, spine.itl2_code))
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1. Fetch from Nomis, paging past the 25,000-row cap

Requests 25,000 rows at a time until a short page signals the end, retrying a page if Nomis drops the connection. Each extract is saved, dated, to `raw/ashe/` and checked to be a complete, duplicate-free grid. Five pulls: the four LA vintages plus every crosswalk predecessor code (~2–3 min in total).

In [2]:
def read_page(url, tries=3):
    for k in range(tries):
        try:
            return pd.read_csv(url, encoding="utf-8-sig")
        except Exception as e:
            if k == tries - 1: raise
            print(f"  retry ({e})"); time.sleep(5)

KEYS = ["DATE_NAME","GEOGRAPHY_CODE","PAY_NAME","ITEM_NAME","MEASURES_NAME"]

def fetch_nomis_all(url, tag, limit=25000):
    frames, offset = [], 0
    while True:
        part = read_page(f"{url}&recordlimit={limit}&recordoffset={offset}")
        frames.append(part)
        if len(part) < limit:
            break
        offset += limit
    df = pd.concat(frames, ignore_index=True)
    df.columns = [c.strip().upper() for c in df.columns]
    # completeness: every date x geography x pay x item x measure exactly once
    grid = 1
    for k in KEYS: grid *= df[k].nunique()
    assert not df.duplicated(KEYS).any(), f"{tag}: duplicate rows - paging overlapped"
    assert len(df) == grid, f"{tag}: {len(df):,} rows but full grid is {grid:,} - paging lost rows"
    raw_path = RAW_DIR / f"nomis_ashe_{tag}_{date.today().isoformat()}.csv"
    df.to_csv(raw_path, index=False)
    print(f"{tag}: {len(df):,} rows, {df.GEOGRAPHY_CODE.nunique()} areas, "
          f"{df.DATE_NAME.nunique()} years -> {raw_path.name}")
    return df

raws = {VINTAGES[t]: fetch_nomis_all(QUERY.format(geo=t), f"workplace_{t}") for t in VINTAGES}
raws["predecessor code"] = fetch_nomis_all(QUERY.format(geo=",".join(sorted(set(xw.old_code)))),
                                           "workplace_predecessors")

workplace_TYPE424: 487,200 rows, 350 areas, 29 years -> nomis_ashe_workplace_TYPE424_2026-10-07.csv


workplace_TYPE432: 505,296 rows, 363 areas, 29 years -> nomis_ashe_workplace_TYPE432_2026-10-07.csv


workplace_TYPE434: 516,432 rows, 371 areas, 29 years -> nomis_ashe_workplace_TYPE434_2026-10-07.csv


workplace_TYPE464: 565,152 rows, 406 areas, 29 years -> nomis_ashe_workplace_TYPE464_2026-10-07.csv


workplace_predecessors: 132,240 rows, 95 areas, 29 years -> nomis_ashe_workplace_predecessors_2026-10-07.csv


## 2. Tidy to an LA-level table

One row per LA unit (as published) × year × pay type × item, with `value`, `cv` and Nomis `status`. **Suppressed cells stay in as NaN** (a `pivot_table` would silently drop them). Each row carries `la_code_current` (crosswalk) and `itl2_code` (spine), plus `geo_vintage` saying which pull it came from.

In [3]:
idx = ["la_code","la_name","year","pay_type","item"]

def tidy(raw, vintage):
    df = raw.rename(columns={"GEOGRAPHY_CODE":"la_code","GEOGRAPHY_NAME":"la_name","PAY_NAME":"pay_type",
                             "ITEM_NAME":"item","OBS_VALUE":"value","OBS_STATUS":"status","MEASURES_NAME":"measure"})
    df["year"]     = df["DATE_NAME"].astype(str).str.extract(r"(\d{4})")[0].astype(int)
    df["la_code"]  = df["la_code"].astype(str).str.strip()
    df["la_name"]  = df["la_name"].str.replace(r"\s*\(.*\)$", "", regex=True).str.strip()   # drop "(Apr 2019 merged ...)"
    df["value"]    = pd.to_numeric(df["value"], errors="coerce")
    df["pay_type"] = df["pay_type"].str.replace(" pay - gross","",regex=False).str.strip()
    df["measure"]  = df["measure"].str.strip().str.lower().map({"value":"value", "confidence":"cv"})
    assert df["measure"].notna().all(), "unexpected measure name from Nomis"
    out = df.set_index(idx + ["measure"])["value"].unstack("measure").reset_index()   # keeps NaN cells
    out = out.join(df[df.measure == "value"].set_index(idx)["status"], on=idx)
    out.columns.name = None
    out["la_code_current"] = out["la_code"].replace(remap)
    out["itl2_code"]       = out["la_code_current"].map(lad2itl)
    out["geo_vintage"]     = vintage
    return out

tidied = {v: tidy(r, v) for v, r in raws.items()}
for v, t in tidied.items():
    bad = sorted(t.loc[t.itl2_code.isna(), "la_code"].unique())
    print(f"{v:17s}: {t.la_code.nunique()} codes | no ITL2 via crosswalk+spine: {bad or 'none'}")

LAD 2023         : 350 codes | no ITL2 via crosswalk+spine: none
LAD 2021         : 363 codes | no ITL2 via crosswalk+spine: none
LAD 2019         : 371 codes | no ITL2 via crosswalk+spine: none
LAD pre-2015     : 406 codes | no ITL2 via crosswalk+spine: ['95AA', '95BB', '95CC', '95DD', '95EE', '95FF', '95GG', '95HH', '95II', '95JJ', '95KK', '95LL', '95MM', '95NN', '95OO', '95PP', '95QQ', '95RR', '95SS', '95TT', '95UU', '95VV', '95WW', '95XX', '95YY', '95ZZ']
predecessor code : 95 codes | no ITL2 via crosswalk+spine: none


### 2b. Stitch the vintages

1. **Same code, several vintages** → report any disagreement, keep the newest typed vintage that has data for that code-year (the code-queried predecessor pull ranks last).
2. **Per current LA × year** → the current LA itself if it has data, else its predecessors (never both).
3. **Predecessors duplicating each other** (identical published values, or the same area under successive codes) → kept once.

Checks: no double coverage, no duplicate rows.

In [4]:
key  = ["la_code","year","pay_type","item"]
prio = {v: i for i, v in enumerate(tidied)}                 # 0 = newest vintage
allv = pd.concat(tidied.values(), ignore_index=True)
allv["prio"] = allv.geo_vintage.map(prio)

# (1) same code-year in several vintages: values must agree; keep newest vintage with data
both = allv.dropna(subset=["value"]).groupby(key).value.agg(["min","max"])
dis = both[both["max"] > both["min"]].reset_index()
print(f"Cells where vintages disagree for the same code-year: {len(dis)}  (resolved by vintage priority)")
if len(dis):
    print(dis.groupby("la_code").year.agg(lambda s: f"{s.min()}-{s.max()}").to_string())
unit_has = allv.groupby(["la_code","year","geo_vintage","prio"]).value.count().reset_index(name="n")
best = (unit_has[unit_has.n > 0].sort_values("prio").drop_duplicates(["la_code","year"])
        [["la_code","year","geo_vintage"]])
units = allv.merge(best, on=["la_code","year","geo_vintage"])

# (2) per current LA-year: itself if it has data, else predecessors
self_years = set(units.loc[units.la_code == units.la_code_current, ["la_code","year"]].itertuples(index=False, name=None))
units = units[(units.la_code == units.la_code_current) |
              pd.Series([ (c, y) not in self_years for c, y in zip(units.la_code_current, units.year) ], index=units.index)]

# (3) predecessors that duplicate each other: identical values, or same area under successive codes
vec = (units.set_index(["la_code_current","year","la_code","la_name","prio","pay_type","item"])["value"]
            .unstack(["pay_type","item"]))
vec = vec.round(2).astype(str).agg("|".join, axis=1).rename("sig").reset_index()
vec = vec.sort_values(["prio","la_code"])
dupe = vec.duplicated(["la_code_current","year","sig"]) | vec.duplicated(["la_code_current","year","la_name"])
dropped = vec.loc[dupe, ["la_code_current","year","la_code","la_name"]]
print(f"Duplicate predecessor units dropped: {len(dropped)}")
print(dropped.groupby(["la_code_current","la_code","la_name"]).year
             .agg(lambda s: f"{s.min()}-{s.max()} ({len(s)} yrs)").to_string())
drop_keys = set(dropped[["la_code","year"]].itertuples(index=False, name=None))
ashe = units[[ (l, y) not in drop_keys for l, y in zip(units.la_code, units.year) ]].drop(columns="prio")

# report gap-fills
pred_used = ashe[(ashe.la_code != ashe.la_code_current)].dropna(subset=["value"])
print("\nCurrent LAs filled from predecessors:")
print(pred_used.groupby("la_code_current").agg(units=("la_code","nunique"),
      years=("year", lambda s: f"{s.min()}-{s.max()}")).to_string())

# checks: no double coverage, no duplicate rows
has = ashe.dropna(subset=["value"])
lvl = has.assign(is_self=has.la_code == has.la_code_current).groupby(["la_code_current","year"]).is_self.nunique()
assert (lvl == 1).all(), f"double coverage: {lvl[lvl > 1].index.tolist()[:5]}"
assert not ashe.duplicated(key).any(), "duplicate LA-year-pay-item rows"

ashe["cv_band"] = pd.cut(ashe["cv"], [0, 5, 10, 20], labels=["precise","reasonably precise","use with caution"])
cols = ["la_code","la_name","la_code_current","itl2_code","geo_vintage","year","pay_type","item",
        "value","cv","cv_band","status"]
ashe = ashe[cols].sort_values(["year","itl2_code","la_code","pay_type","item"]).reset_index(drop=True)
ashe = ashe[ashe.year.isin(ashe.dropna(subset=["value"]).year.unique())]

OUT.parent.mkdir(parents=True, exist_ok=True)
ashe.to_csv(OUT, index=False)
print(f"\nSaved -> {OUT}  ({len(ashe):,} rows, {ashe.year.min()}-{ashe.year.max()})")
ashe.head(12)

Cells where vintages disagree for the same code-year: 7293  (resolved by vintage priority)
la_code
E06000001    2022-2022
E06000002    2022-2022
E06000003    2022-2022
E06000004    2022-2022
E06000005    2022-2022
E06000006    2022-2022
E06000007    2022-2022
E06000008    2022-2022
E06000009    2022-2022
E06000010    2022-2022
E06000011    2022-2022
E06000012    2022-2022
E06000013    2022-2022
E06000014    2022-2022
E06000015    2022-2022
E06000016    2022-2022
E06000017    2022-2022
E06000018    2022-2022
E06000019    2022-2022
E06000020    2022-2022
E06000021    2022-2022
E06000022    2022-2022
E06000023    2022-2022
E06000024    2022-2022
E06000025    2022-2022
E06000026    2022-2022
E06000027    2022-2022
E06000030    2022-2022
E06000031    2022-2022
E06000032    2022-2022
E06000033    2022-2022
E06000034    2022-2022
E06000035    2022-2022
E06000036    2022-2022
E06000037    2022-2022
E06000038    2022-2022
E06000039    2022-2022
E06000040    2022-2022
E06000041    2022-2022
E060

Duplicate predecessor units dropped: 0
Series([], )

Current LAs filled from predecessors:
                 units      years
la_code_current                  
E06000049            3  1997-2007
E06000050            3  1997-2007
E06000052            6  1997-2007
E06000056            2  1997-2007
E06000058            3  1997-2017
E06000059            5  1997-2007
E06000060            4  1997-2007
E06000061            4  1997-2019
E06000062            3  1997-2019
E06000063            3  1997-2021
E06000064            3  1997-2021
E06000065            7  1997-2007
E06000066            5  1997-2007
E07000244            2  1997-2017
E07000245            2  1997-2017
E08000038            1  1997-2025
E08000039            1  1997-2025



Saved -> /Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel/clean/ashe_workplace_lad.csv  (257,304 rows, 1997-2025)


,la_code,la_name,la_code_current,itl2_code,geo_vintage,year,pay_type,item,value,cv,cv_band,status
0,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,10 percentile,4.08,6.2,reasonably precise,A
1,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,20 percentile,4.81,6.9,reasonably precise,A
2,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,25 percentile,5.05,7.2,reasonably precise,A
3,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,30 percentile,5.59,6.8,reasonably precise,A
4,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,40 percentile,6.29,8.5,reasonably precise,A
5,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,60 percentile,8.23,10.0,reasonably precise,A
6,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,70 percentile,9.23,13.0,use with caution,A
7,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,75 percentile,10.18,16.0,use with caution,A
8,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,80 percentile,NaN,NaN,NaN,B
9,E06000001,Hartlepool,E06000001,TLC3,LAD 2019,1997,Hourly,90 percentile,14.02,0.8,precise,A


## 3. Coverage, reliability & sanity checks

Coverage by year (units and ITL2 regions with a median), suppression by item, CV bands, then internal-consistency checks that should all pass: percentiles ordered P10 ≤ … ≤ P90, the same jobs count for weekly and hourly, implied full-time weekly hours (weekly ÷ hourly median) in a plausible ~35–45 range, and a list of large year-on-year jumps per unit (small-sample areas — treat with care). Year-on-year changes are only computed within a geography vintage.

In [5]:
med_w = ashe[(ashe.item == "Median") & (ashe.pay_type == "Weekly")]
cov = med_w.dropna(subset=["value"]).groupby("year").agg(units=("la_code","nunique"), itl2=("itl2_code","nunique"))
print("Units / ITL2 regions with a weekly median, by year (GB has 45 ITL2):")
print(cov.T.to_string())
gb_itl2 = set(spine.itl2_code) - {"TLN0"}
for y, g in med_w.dropna(subset=["value"]).groupby("year"):
    miss = gb_itl2 - set(g.itl2_code)
    if miss: print(f"  {y}: ITL2 with no LA median: {sorted(miss)}")
cur_cov = med_w.dropna(subset=["value"]).groupby("year").la_code_current.nunique()
print("\nCurrent LAs with a weekly median (itself or via predecessors), by year (GB = 350):")
print(cur_cov.to_frame().T.to_string())
for y, g in med_w.dropna(subset=["value"]).groupby("year"):
    miss = sorted(set(spine.la_code[~spine.la_code.str.startswith("N")]) - set(g.la_code_current))
    if miss: print(f"  {y}: current LAs with no median: {miss}")
print("Spine LAs never in ASHE:", sorted(set(spine.la_code) - set(ashe.la_code_current)), "(NI expected)")

print(f"\nSuppressed (no value): {ashe.value.isna().sum():,} of {len(ashe):,} cells. Status counts:")
print(ashe.status.value_counts().to_string())
share = (ashe[ashe.year.isin([2005, ashe.year.max()])].groupby(["item","year"]).value
             .apply(lambda s: s.isna().mean()).unstack().mul(100).round(0))
print("\n% of units suppressed (weekly+hourly pooled):"); print(share.to_string())
print("\nCV bands (non-missing cells):"); print(ashe.cv_band.value_counts().to_string())

# --- internal consistency ---
ORDER = ["10 percentile","20 percentile","25 percentile","30 percentile","40 percentile","Median",
         "60 percentile","70 percentile","75 percentile","80 percentile","90 percentile"]
w = ashe.pivot_table(index=["la_code","geo_vintage","year","pay_type"], columns="item", values="value")
bad = (w[ORDER].diff(axis=1) < 0).any(axis=1)
print(f"\nPercentile-order violations: {bad.sum()}")

jobs = w["Number of jobs"].unstack("pay_type")
print(f"Jobs weekly vs hourly, max abs difference: {(jobs.Weekly - jobs.Hourly).abs().max():.0f}")

med = w["Median"].unstack("pay_type")
hours = (med.Weekly / med.Hourly).dropna()
print(f"Implied FT weekly hours (weekly/hourly median): min {hours.min():.1f}, median {hours.median():.1f}, max {hours.max():.1f}")

yoy = (med.Weekly.unstack("year").pct_change(axis=1, fill_method=None).mul(100).stack())
print("\nWeekly median |YoY| > 25% (check before use):")
print(yoy[yoy.abs() > 25].round(1).to_string())

uk = med_w.groupby("year").value.median()
print("\nMedian across units of the weekly median (£) — watch 2004/2006/2011/2021 breaks:")
print(uk.round(0).to_frame().T.to_string())

Units / ITL2 regions with a weekly median, by year (GB has 45 ITL2):
year   1997  1998  1999  2000  2001  2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
units   388   389   387   387   388   387   386   388   387   389   387   360   363   362   362   361   362   362   362   362   362   358   359   353   353   349   349   349   349
itl2     45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45    45

Current LAs with a weekly median (itself or via predecessors), by year (GB = 350):
year             1997  1998  1999  2000  2001  2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
la_code_current   349   350   348   348   349   348   347   349   348   349   348   347   350   349   349   348   349


Percentile-order violations: 0
Jobs weekly vs hourly, max abs difference: 0
Implied FT weekly hours (weekly/hourly median): min 35.1, median 39.9, max 50.3

Weekly median |YoY| > 25% (check before use):
la_code    geo_vintage   year
E06000003  LAD 2019      2004    27.2
E06000028  LAD pre-2015  2007    26.3
E06000040  LAD 2019      1999    25.1
E07000027  LAD 2019      2001    25.1
E07000064  LAD 2019      2001    33.5
           LAD 2023      2021    25.4
E07000074  LAD 2019      2001    25.5
           LAD 2023      2022    26.0
E07000079  LAD 2019      2004    25.6
E07000109  LAD 2023      2019    26.2
E07000120  LAD 2019      2002    31.8
E07000122  LAD 2019      2000    25.7
E07000133  LAD 2023      2022    33.3
E07000173  LAD 2023      2012    32.9
E07000203  LAD 2019      2001    29.4
E07000210  LAD 2023      2023    26.3
E07000217  LAD 2019      1998    29.5
E07000221  LAD 2023      2025    33.1
E07000223  LAD 2019      2007    33.7
E07000235  LAD 2019      1999    25.3
E07000